# 06_Unsupervised_Model_Development_LDA

In [10]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import LatentDirichletAllocation
import polars as pl


In [2]:
# Load the parquet file
INPUT_PATH = '../data/processed/clustered_narratives.parquet'
df_narratives = pd.read_parquet(
    INPUT_PATH,
    columns=[
        'Complaint ID',
        'processed_narrative'
    ]
)

In [6]:
df_narratives.head()

,Complaint ID,processed_narrative
0,3442136,claimed delivered package address never receiv...
1,3601853,got brink money pre paid card mail assuming un...
2,3300820,called creditor nelson cruz associate claimed ...
3,3739698,around opened credit card account online capit...
4,3619130,rushmore loan management permit applying loan ...


In [3]:
# Run Count Vectorizer on 50k sample to save memory
df_lda_sample = df_narratives.sample(
    n=50_000,
    random_state=42
)

count_vectorizer = CountVectorizer(
    min_df=20,
    max_df=0.90,
    max_features=10_000
)

X_count = count_vectorizer.fit_transform(
    df_lda_sample['processed_narrative']
)

print(X_count.shape)

(50000, 6455)


In [4]:
# Run LDA on sample
lda = LatentDirichletAllocation(
    n_components=10,
    learning_method='online',
    random_state=42,
    n_jobs=-1,
    batch_size=2048
)

lda.fit(X_count)

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",10
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'online'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",2048
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


In [5]:
feature_names = np.array(count_vectorizer.get_feature_names_out())

for topic_idx, topic in enumerate(lda.components_):
    top_indices = topic.argsort()[::-1][:15]
    top_terms = feature_names[top_indices]

    print(f'\nTopic {topic_idx}')
    print(', '.join(top_terms))


Topic 0
account, complaint, financial, credit, bank, request, information, consumer, provide, issue, matter, despite, action, regarding, without

Topic 1
charge, dispute, transaction, claim, card, chase, merchant, refund, received, purchase, made, amount, fraud, time, service

Topic 2
payment, fee, late, interest, balance, account, month, due, pay, statement, paid, made, amount, charge, time

Topic 3
account, bank, money, told, would, called, call, back, said, day, time, get, could, phone, number

Topic 4
account, credit, one, capital, report, information, card, closed, number, address, name, fraud, opened, identity, never

Topic 5
card, credit, would, time, citi, told, called, account, customer, received, purchase, service, point, citibank, said

Topic 6
well, fargo, loan, mortgage, modification, foreclosure, document, home, sale, property, letter, attorney, court, servicing, complaint

Topic 7
check, account, fund, bank, fee, deposit, transaction, overdraft, balance, checking, trans

In [6]:
topic_probs = lda.transform(X_count)

df_lda_sample['dominant_topic'] = topic_probs.argmax(axis=1)
df_lda_sample['topic_probability'] = topic_probs.max(axis=1)

df_lda_sample['dominant_topic'].value_counts().sort_index()

dominant_topic
0     3565
1     4962
2     5283
3    12950
4     3820
5     5602
6     2527
7     2889
8     7317
9     1085
Name: count, dtype: int64

In [8]:
for topic in sorted(df_lda_sample['dominant_topic'].unique()):
    print(f'\n===== Topic {topic} =====')

    samples = (
        df_lda_sample[df_lda_sample['dominant_topic'] == topic]
        .sample(n=3, random_state=42)
    )

    for text in samples['processed_narrative']:
        print('\n', text[:500])


===== Topic 0 =====

 dear fraud investigation department writing formally demand elan financial service provide immediate complete documentation resolution concerning fraudulent charge account ending despite prior confirmation fraud elan reversed credit following transaction year year previously deemed fraudulent account closed result reversal provisional credit without transparent investigation explanation direct violation regulation cfr fair credit billing act electronic fund transfer act additionally transaction 

 formal complaint regarding unauthorized charge request assistance dear consumer financial protection bureau cfpb writing formally file complaint regarding multiple unauthorized charge made bank account year totaling reported issue promptly truist bank provided requested documentation support investigation however today received refund clear resolution deadline established electronic fund transfer act regulation exceeded given lack action communication bank seeking cfpbs

In [11]:
# Bring in sample of cleaned narratives
df_small = (
    pl.scan_parquet(INPUT_PATH)
    .select([
        'cleaned_consumer_narrative',
        'processed_narrative'
    ])
    .sample(n=5000, seed=42)
    .collect()
    .to_pandas()
)

print(df_small.shape)
df_small.head()

AttributeError: 'LazyFrame' object has no attribute 'sample'

In [14]:
sample_ids = df_lda_sample['Complaint ID'].to_list()

df_readable = (
    pl.scan_parquet(INPUT_PATH)
    .filter(pl.col('Complaint ID').is_in(sample_ids))
    .select([
        'Complaint ID',
        'cleaned_consumer_narrative'
    ])
    .collect()
    .to_pandas()
)

df_lda_sample = df_lda_sample.merge(
    df_readable,
    on='Complaint ID',
    how='left'
)

In [15]:
for topic in sorted(df_lda_sample['dominant_topic'].unique()):
    print(f'\n===== Topic {topic} =====')

    samples = (
        df_lda_sample[df_lda_sample['dominant_topic'] == topic]
        .sample(n=3, random_state=42)
    )

    for text in samples['cleaned_consumer_narrative']:
        print('\n', text[:500])


===== Topic 0 =====

 Dear Fraud Investigation Department, I am writing to formally demand that Elan Financial Services provide immediate and complete documentation and resolution concerning fraudulent charges on my accounts ending in REDACTED and REDACTED . Despite prior confirmations of fraud, Elan reversed credits for the following transactions : - {$620.00} at REDACTED # REDACTED on REDACTED / REDACTED /year> - {$2000.00} at REDACTED on REDACTED / REDACTED /year> These were previously deemed fraudulent, and my ac

 Formal Complaint Regarding Unauthorized Charges Request for Assistance Dear Consumer Financial Protection Bureau ( CFPB ), I am writing to formally file a complaint regarding multiple unauthorized charges made to my bank account between REDACTED_DATE and REDACTED / REDACTED /year>, totaling over {$40000.00}. I reported the issue promptly to Truist Bank and provided all requested documentation to support their investigation. However, as of today, I have not received a re